# Этап 2. Baseline и Content-Based рекомендации

## Цель работы

Реализовать базовые подходы к построению книжных рекомендаций
и исследовать их особенности.

В рамках этапа необходимо:

1. Подготовить данные для построения рекомендательных моделей.
2. Реализовать Popularity Baseline на основе популярности книг.
3. Сформировать текстовые признаки книг из названий и тегов.
4. Применить TF-IDF для векторизации текстовых представлений.
5. Рассчитать косинусную близость между книгами.
6. Реализовать функцию `get_similar_books()`.
7. Проанализировать примеры рекомендаций и ограничения моделей.


## 2.1. Подготовка данных

Перед построением рекомендательных моделей подготовим пользовательские оценки.

На этапе EDA было обнаружено 2 278 избыточных записей по ключу `(user_id, book_id)`.
Некоторые повторяющиеся пары имеют разные значения рейтинга.

Для устранения повторений объединим оценки одного пользователя для одной книги,
рассчитав их среднее арифметическое.

Исходные данные в `data/raw/` оставляем без изменений.

In [1]:
# === Импорты ===

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from homework_tasks.recommendation_systems.homework_goodbooks.src.data_loader import (
    load_books,
    load_ratings,
    load_book_tags,
    load_tags,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.data_preparation import (
    prepare_ratings,
    split_ratings,
    save_ratings_split,
    prepare_book_content,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.recommenders import (
    build_popularity_ranking,
    get_popular_recommendations,
    get_similar_books,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.config import (
    MAX_TAGS_PER_BOOK,
)

In [2]:
# === Загрузка исходных данных ===

ratings = load_ratings()
books = load_books()

print(f"Исходное количество оценок: {len(ratings)}")
print(f"Количество книг: {len(books)}")

# === Обработка повторяющихся оценок ===

prepared_ratings = prepare_ratings(ratings)

print(f"Количество оценок после подготовки: {len(prepared_ratings)}")
print(f"Удалено избыточных записей: {len(ratings) - len(prepared_ratings)}")

# Проверяем уникальность пар пользователь–книга
duplicates_count = prepared_ratings.duplicated(
    subset=["user_id", "book_id"]
).sum()

print(f"Повторяющихся пар после подготовки: {duplicates_count}")

display(prepared_ratings.head())

Исходное количество оценок: 981756
Количество книг: 10000
Количество оценок после подготовки: 979478
Удалено избыточных записей: 2278
Повторяющихся пар после подготовки: 0


,user_id,book_id,rating
0,1,1180,4.0
1,1,4893,3.0
2,1,6285,4.0
3,2,8034,4.0
4,2,8855,5.0


### Разделение данных на обучающую и тестовую выборки

Для последующего обучения и сравнения рекомендательных моделей разделим подготовленные оценки на train и test.

Разделение выполняется отдельно для каждого пользователя:

- около 80% взаимодействий используются для обучения
- около 20% для оценки качества моделей
- в обучающей выборке сохраняется не менее одной оценки каждого пользователя

Поскольку временные метки отсутствуют, используем случайное разбиение с фиксированным `random_state=42`.

Полученное разбиение будет общим для Popularity Baseline, Item-Based Collaborative Filtering и SVD.
Это позволит сравнивать модели на одинаковых данных.

In [3]:
# === Разделение оценок на train и test ===

train_ratings, test_ratings = split_ratings(prepared_ratings)

print(f"Обучающая выборка: {train_ratings.shape}")
print(f"Тестовая выборка: {test_ratings.shape}")

print(f"Доля train: {len(train_ratings) / len(prepared_ratings):.2%}")
print(f"Доля test: {len(test_ratings) / len(prepared_ratings):.2%}")

# Проверяем, что пользователи из test присутствуют в train
unknown_users = set(test_ratings["user_id"]) - set(train_ratings["user_id"])

print(f"Неизвестных пользователей в test: {len(unknown_users)}")

# Проверяем отсутствие общих взаимодействий между выборками
common_interactions = train_ratings.merge(
    test_ratings,
    on=["user_id", "book_id"],
)

print(f"Пересечений между train и test: {len(common_interactions)}")


Обучающая выборка: (788503, 3)
Тестовая выборка: (190975, 3)
Доля train: 80.50%
Доля test: 19.50%
Неизвестных пользователей в test: 0
Пересечений между train и test: 0


In [4]:
# === Статистика пользователей и книг после разделения ===

split_stats = pd.DataFrame(
    {
        "Показатель": [
            "Количество взаимодействий",
            "Количество пользователей",
            "Количество книг",
            "Средняя оценка",
        ],
        "Train": [
            len(train_ratings),
            train_ratings["user_id"].nunique(),
            train_ratings["book_id"].nunique(),
            round(train_ratings["rating"].mean(), 2),
        ],
        "Test": [
            len(test_ratings),
            test_ratings["user_id"].nunique(),
            test_ratings["book_id"].nunique(),
            round(test_ratings["rating"].mean(), 2),
        ],
    }
)

display(split_stats)

,Показатель,Train,Test
0,Количество взаимодействий,788503.00,190975.00
1,Количество пользователей,53424.00,53382.00
2,Количество книг,10000.00,10000.00
3,Средняя оценка,3.86,3.86


### Выводы по подготовке данных

По результатам подготовки и разделения пользовательских оценок установлено:

1. После агрегации повторяющихся оценок количество взаимодействий уменьшилось с 981 756 до 979 478.
Повторяющихся пар `(user_id, book_id)` не осталось.

2. Подготовленные данные разделены на обучающую и тестовую выборки
в соотношении 80,50% и 19,50% соответственно.

3. В обучающей выборке представлены все 53 424 пользователя и 10 000 книг.
В тестовой выборке содержатся 53 382 пользователя и все 10 000 книг.

4. Все пользователи тестовой выборки присутствуют в обучающей.
Пересечений между train и test по парам пользователь - книга не обнаружено.

5. Средняя оценка в обеих выборках составляет 3,86,
что указывает на близость распределений по среднему значению.

6. Разделение выполнено случайным образом внутри пользователей с фиксированным `random_state=42`.
Поскольку временные метки отсутствуют, такое разбиение не учитывает хронологию взаимодействий.

**Итог:**

Подготовлены единые обучающая и тестовая выборки для построения
и последующего сравнения рекомендательных моделей.

Обучение моделей и расчет статистики взаимодействий будут выполняться только на train.
Тестовая выборка будет использоваться для оценки качества рекомендаций.

In [5]:
# === Сохранение подготовленных выборок ===

save_ratings_split(
    train_ratings=train_ratings,
    test_ratings=test_ratings,
)


Обучающая выборка сохранена: /Users/pontigor/python_test/mephi_homework_tasks/homework_tasks/recommendation_systems/homework_goodbooks/data/processed/train_ratings.csv
Тестовая выборка сохранена: /Users/pontigor/python_test/mephi_homework_tasks/homework_tasks/recommendation_systems/homework_goodbooks/data/processed/test_ratings.csv


## 2.2. Popularity Baseline

Реализуем базовую рекомендательную модель,
которая предлагает книги с наиболее высокими средними оценками.

Для построения рекомендаций используем:

- среднюю пользовательскую оценку книги
- количество полученных оценок
- минимальный порог количества оценок

Минимальный порог необходим, чтобы исключить книги
с недостаточным количеством наблюдений.

Все показатели рассчитываются только по обучающей выборке,
чтобы не использовать тестовые взаимодействия.

Данный подход не учитывает индивидуальные предпочтения пользователей
и формирует общий список рекомендуемых книг.

In [6]:
# === Расчет статистики популярности книг ===

# Минимальное количество оценок для участия в рейтинге
MIN_RATINGS_COUNT = 20

book_rating_stats = (
    train_ratings.groupby("book_id")
    .agg(
        ratings_count=("rating", "count"),
        average_rating=("rating", "mean"),
    )
    .reset_index()
)

# Исключаем книги с недостаточным количеством оценок
popular_books = book_rating_stats[
    book_rating_stats["ratings_count"] >= MIN_RATINGS_COUNT
    ].copy()

# Сортируем по средней оценке,
# а при равенстве - по количеству оценок
popular_books = popular_books.sort_values(
    ["average_rating", "ratings_count", "book_id"],
    ascending=[False, False, True],
)

print(f"Всего книг в train: {len(book_rating_stats)}")
print(f"Книг после фильтрации: {len(popular_books)}")
print(f"Минимальное количество оценок: {MIN_RATINGS_COUNT}")

display(popular_books.head(10))

Всего книг в train: 10000
Книг после фильтрации: 9997
Минимальное количество оценок: 20


,book_id,ratings_count,average_rating
6919,6920,73,4.835616
7946,7947,67,4.820896
5579,5580,85,4.811765
8662,8663,82,4.792683
3274,3275,86,4.790698
9565,9566,80,4.775000
4482,4483,85,4.764706
5206,5207,80,4.762500
6360,6361,76,4.750000
1307,1308,82,4.743902


In [7]:
# === Топ-10 книг по Popularity Baseline ===

top_popular_recommendations = popular_books.head(10).merge(
    books[["id", "title", "authors"]],
    left_on="book_id",
    right_on="id",
    how="left",
    validate="one_to_one",
)

top_popular_recommendations = top_popular_recommendations[
    ["book_id", "title", "authors", "ratings_count", "average_rating"]
]

display(top_popular_recommendations)

,book_id,title,authors,ratings_count,average_rating
0,6920,The Indispensable Calvin and Hobbes,Bill Watterson,73,4.835616
1,7947,ESV Study Bible,"Anonymous, Lane T. Dennis, Wayne A. Grudem",67,4.820896
2,5580,The Calvin and Hobbes Lazy Sunday Book,Bill Watterson,85,4.811765
3,8663,"Locke & Key, Vol. 6: Alpha & Omega","Joe Hill, Gabriel Rodríguez",82,4.792683
4,3275,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...","J.K. Rowling, Mary GrandPré",86,4.790698
5,9566,Attack of the Deranged Mutant Killer Monster S...,Bill Watterson,80,4.775000
6,4483,It's a Magical World: A Calvin and Hobbes Coll...,Bill Watterson,85,4.764706
7,5207,The Days Are Just Packed: A Calvin and Hobbes ...,Bill Watterson,80,4.762500
8,6361,There's Treasure Everywhere: A Calvin and Hobb...,Bill Watterson,76,4.750000
9,1308,A Court of Mist and Fury (A Court of Thorns an...,Sarah J. Maas,82,4.743902


### Получение рекомендаций Popularity Baseline

Выделим логику расчета рейтинга и формирования рекомендаций в отдельные функции.

Проверим два сценария:

1. Общий список популярных книг без учета пользователя.
2. Рекомендации для конкретного пользователя с исключением книг,
которые он уже оценил в обучающей выборке.

Во втором сценарии рейтинг остается общим.
Персонализация заключается только в исключении ранее оцененных книг.

In [8]:
# === Формирование общего рейтинга через функцию ===

popularity_ranking = build_popularity_ranking(
    train_ratings=train_ratings,
    min_ratings_count=MIN_RATINGS_COUNT,
)

# Проверяем, что результат совпадает с расчетом непосредственно в ноутбуке
assert popularity_ranking["book_id"].tolist() == popular_books["book_id"].tolist()

general_recommendations = get_popular_recommendations(
    popularity_ranking=popularity_ranking,
    books=books,
    top_n=10,
)

display(general_recommendations)

,book_id,title,authors,ratings_count,average_rating
0,6920,The Indispensable Calvin and Hobbes,Bill Watterson,73,4.835616
1,7947,ESV Study Bible,"Anonymous, Lane T. Dennis, Wayne A. Grudem",67,4.820896
2,5580,The Calvin and Hobbes Lazy Sunday Book,Bill Watterson,85,4.811765
3,8663,"Locke & Key, Vol. 6: Alpha & Omega","Joe Hill, Gabriel Rodríguez",82,4.792683
4,3275,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...","J.K. Rowling, Mary GrandPré",86,4.790698
5,9566,Attack of the Deranged Mutant Killer Monster S...,Bill Watterson,80,4.775000
6,4483,It's a Magical World: A Calvin and Hobbes Coll...,Bill Watterson,85,4.764706
7,5207,The Days Are Just Packed: A Calvin and Hobbes ...,Bill Watterson,80,4.762500
8,6361,There's Treasure Everywhere: A Calvin and Hobb...,Bill Watterson,76,4.750000
9,1308,A Court of Mist and Fury (A Court of Thorns an...,Sarah J. Maas,82,4.743902


In [9]:
# === Рекомендации пользователю с учетом истории оценок ===

# Выбираем пользователя, который уже оценил самую популярную книгу
top_book_id = int(popularity_ranking.iloc[0]["book_id"])

user_id = int(
    train_ratings.loc[
        train_ratings["book_id"] == top_book_id,
        "user_id",
    ].iloc[0]
)

user_recommendations = get_popular_recommendations(
    popularity_ranking=popularity_ranking,
    books=books,
    top_n=10,
    user_id=user_id,
    train_ratings=train_ratings,
)

rated_book_ids = set(
    train_ratings.loc[
        train_ratings["user_id"] == user_id,
        "book_id",
    ]
)

recommended_book_ids = set(user_recommendations["book_id"])

print(f"ID пользователя: {user_id}")
print(f"Количество оцененных книг в train: {len(rated_book_ids)}")
print(f"Количество рекомендаций: {len(user_recommendations)}")
print(
    "Уже оцененных книг в рекомендациях:",
    len(recommended_book_ids & rated_book_ids),
)

# Проверяем, что ранее оцененная книга исключена из выдачи
assert top_book_id in rated_book_ids
assert top_book_id not in recommended_book_ids
assert not (recommended_book_ids & rated_book_ids)

display(user_recommendations)


ID пользователя: 6147
Количество оцененных книг в train: 19
Количество рекомендаций: 10
Уже оцененных книг в рекомендациях: 0


,book_id,title,authors,ratings_count,average_rating
0,7947,ESV Study Bible,"Anonymous, Lane T. Dennis, Wayne A. Grudem",67,4.820896
1,5580,The Calvin and Hobbes Lazy Sunday Book,Bill Watterson,85,4.811765
2,8663,"Locke & Key, Vol. 6: Alpha & Omega","Joe Hill, Gabriel Rodríguez",82,4.792683
3,3275,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...","J.K. Rowling, Mary GrandPré",86,4.790698
4,9566,Attack of the Deranged Mutant Killer Monster S...,Bill Watterson,80,4.775000
5,5207,The Days Are Just Packed: A Calvin and Hobbes ...,Bill Watterson,80,4.762500
6,6361,There's Treasure Everywhere: A Calvin and Hobb...,Bill Watterson,76,4.750000
7,1308,A Court of Mist and Fury (A Court of Thorns an...,Sarah J. Maas,82,4.743902
8,4868,Jesus the Christ,James E. Talmage,82,4.743902
9,8109,"The Absolute Sandman, Volume One","Neil Gaiman, Mike Dringenberg, Chris Bachalo, ...",78,4.743590


### Выводы по Popularity Baseline

По результатам реализации базовой рекомендательной модели установлено:

1. Рейтинг книг формируется на основе средней пользовательской оценки
с дополнительным ограничением по минимальному количеству оценок.

2. При минимальном пороге в 20 оценок в рейтинге осталось 9 997 книг из 10 000.
Таким образом, фильтрация исключила только 3 книги.

3. В верхней части рейтинга представлены книги с высокими средними оценками,
включая несколько сборников комиксов Calvin and Hobbes.

4. Модель формирует одинаковый рейтинг для всех пользователей,
поскольку не учитывает их индивидуальные предпочтения.

5. Для конкретного пользователя реализовано исключение книг,
которые он уже оценил в обучающей выборке.
Это позволяет избежать повторных рекомендаций известных книг.

6. Основными преимуществами Popularity Baseline являются простота реализации,
высокая скорость формирования рекомендаций и возможность использования
при отсутствии пользовательской истории.

7. Основные ограничения модели - отсутствие содержательной персонализации,
зависимость от количества и распределения оценок,
а также возможное смещение рекомендаций в сторону высоко оцененных произведений.

**Итог:**

Реализована простая базовая модель книжных рекомендаций,
которая будет использоваться для сравнения с более сложными алгоритмами.

## 2.3. Подготовка текстовых признаков книг

Для построения Content-Based рекомендаций необходимо сформировать текстовое представление каждой книги.

В качестве источников информации используем:

- `original_title` -> оригинальное название книги
- `title` -> название книги, используемое при отсутствии оригинального
- пользовательские теги -> дополнительные характеристики жанра и содержания произведения

На этапе EDA было обнаружено, что некоторые названия отсутствуют,
а значительная часть тегов описывает действия пользователей,
а не содержание книг.

Поэтому перед применением TF-IDF необходимо подготовить названия
и отобрать наиболее информативные теги.

In [10]:
# === Подготовка названий книг ===

books_content = books[
    ["id", "book_id", "title", "original_title", "authors"]
].copy()

# При отсутствии оригинального названия используем основное название книги
books_content["book_title"] = books_content["original_title"].fillna(
    books_content["title"]
)

print(f"Количество книг: {len(books_content)}")
print(
    "Книг без подготовленного названия:",
    books_content["book_title"].isna().sum(),
)

display(books_content.head(10))

Количество книг: 10000
Книг без подготовленного названия: 0


,id,book_id,title,original_title,authors,book_title
0,1,2767052,"The Hunger Games (The Hunger Games, #1)",The Hunger Games,Suzanne Collins,The Hunger Games
1,2,3,Harry Potter and the Sorcerer's Stone (Harry P...,Harry Potter and the Philosopher's Stone,"J.K. Rowling, Mary GrandPré",Harry Potter and the Philosopher's Stone
2,3,41865,"Twilight (Twilight, #1)",Twilight,Stephenie Meyer,Twilight
3,4,2657,To Kill a Mockingbird,To Kill a Mockingbird,Harper Lee,To Kill a Mockingbird
4,5,4671,The Great Gatsby,The Great Gatsby,F. Scott Fitzgerald,The Great Gatsby
5,6,11870085,The Fault in Our Stars,The Fault in Our Stars,John Green,The Fault in Our Stars
6,7,5907,The Hobbit,The Hobbit or There and Back Again,J.R.R. Tolkien,The Hobbit or There and Back Again
7,8,5107,The Catcher in the Rye,The Catcher in the Rye,J.D. Salinger,The Catcher in the Rye
8,9,960,"Angels & Demons (Robert Langdon, #1)",Angels & Demons,Dan Brown,Angels & Demons
9,10,1885,Pride and Prejudice,Pride and Prejudice,Jane Austen,Pride and Prejudice


In [11]:
# === Фильтрация служебных тегов ===

SERVICE_TAGS = {
    "to-read",
    "currently-reading",
    "read",
    "favorites",
    "owned",
    "books-i-own",
    "kindle",
    "ebook",
    "audiobook",
}

book_tags_with_names = load_book_tags().merge(
    load_tags(),
    on="tag_id",
    how="left",
    validate="many_to_one",
)

# Исключаем служебные метки из текстовых характеристик книг
content_tags = book_tags_with_names[
    ~book_tags_with_names["tag_name"].isin(SERVICE_TAGS)
].copy()

print(f"Исходное количество связей книга–тег: {len(book_tags_with_names)}")
print(f"После фильтрации: {len(content_tags)}")
print(
    f"Количество исключенных записей: {len(book_tags_with_names) - len(content_tags)}")

display(content_tags.head(10))

Исходное количество связей книга–тег: 999912
После фильтрации: 927003
Количество исключенных записей: 72909


,goodreads_book_id,tag_id,count,tag_name
1,1,11305,37174,fantasy
4,1,33114,12716,young-adult
5,1,11743,9954,fiction
6,1,14017,7169,harry-potter
9,1,32989,4364,ya
10,1,27199,3857,series
11,1,11590,3456,favourites
12,1,18886,3374,magic
13,1,6953,2408,childrens
14,1,22753,1881,owned-books


### Проверка информативности тегов после фильтрации

После исключения первоначального списка служебных тегов осталось 927 003 связи между книгами и тегами.

Проверим, какие теги встречаются наиболее часто в оставшихся данных.

Это позволит выявить дополнительные служебные метки и определить,
какие характеристики целесообразно использовать для формирования текстовых профилей книг.

In [12]:
# === Топ-30 тегов после первичной фильтрации ===

top_content_tags = (
    content_tags.groupby("tag_name")
    .agg(
        total_count=("count", "sum"),
        books_count=("goodreads_book_id", "nunique"),
    )
    .sort_values("total_count", ascending=False)
    .head(30)
    .reset_index()
)

display(top_content_tags)

,tag_name,total_count,books_count
0,fiction,3688819,9097
1,fantasy,3548157,4259
2,young-adult,1848306,3630
3,classics,1756920,2785
4,romance,1231926,4251
5,ya,898334,2842
6,mystery,872282,3686
7,non-fiction,857901,2128
8,historical-fiction,815421,2590
9,series,782637,5080


In [13]:
# === Количество уникальных тегов после фильтрации ===

filtered_tags_per_book = (
    content_tags.groupby("goodreads_book_id")["tag_id"]
    .nunique()
)

print(f"Количество книг с тегами: {len(filtered_tags_per_book)}")
print(f"Среднее количество тегов: {filtered_tags_per_book.mean():.2f}")
print(f"Медианное количество тегов: {filtered_tags_per_book.median():.2f}")
print(f"Минимальное количество тегов: {filtered_tags_per_book.min()}")
print(f"Максимальное количество тегов: {filtered_tags_per_book.max()}")

# Проверяем, остались ли книги без тегов после фильтрации
books_without_tags = (
    set(books_content["book_id"])
    - set(content_tags["goodreads_book_id"])
)

print(f"Книг без тегов после фильтрации: {len(books_without_tags)}")

Количество книг с тегами: 10000
Среднее количество тегов: 92.70
Медианное количество тегов: 92.00
Минимальное количество тегов: 51
Максимальное количество тегов: 98
Книг без тегов после фильтрации: 0


### Формирование текстовых профилей книг

Для построения Content-Based модели сформируем единое текстовое представление каждой книги.

Подготовка выполняется следующим образом:

1. Используем `original_title`, а при его отсутствии - `title`.
2. Исключаем служебные теги, не описывающие содержание произведения.
3. Объединяем повторяющиеся связи между книгами и тегами.
4. Выбираем 20 наиболее часто применяемых содержательных тегов для каждой книги.
5. Объединяем название книги и выбранные теги в поле `content_text`.

Полученные текстовые профили будут использоваться для построения TF-IDF матрицы и поиска похожих книг.

In [14]:
# === Формирование текстовых представлений книг ===

# В отличие от первоначальной исследовательской фильтрации,
# итоговая функция prepare_book_content() использует расширенный
# список служебных тегов, сформированный по результатам анализа
books_content = prepare_book_content(
    books=books,
    book_tags=load_book_tags(),
    tags=load_tags(),
    max_tags_per_book=MAX_TAGS_PER_BOOK,
)

print(f"Количество книг: {len(books_content)}")
print(f"Количество уникальных ID: {books_content['id'].nunique()}")
print(f"Книг без текстового профиля: {(books_content['content_text'] == '').sum()}")

# Проверяем ограничение количества выбранных тегов
selected_tags_count = books_content["tags_text"].str.split().str.len()

print(f"Максимальное количество тегов/слов по пробелам: {selected_tags_count.max()}")

display(
    books_content[
        ["id", "title", "book_title", "tags_text", "content_text"]
    ].head()
)

Количество книг: 10000
Количество уникальных ID: 10000
Книг без текстового профиля: 0
Максимальное количество тегов/слов по пробелам: 20


,id,title,book_title,tags_text,content_text
0,1,"The Hunger Games (The Hunger Games, #1)",The Hunger Games,young-adult fiction dystopian dystopia fantasy...,The Hunger Games young-adult fiction dystopian...
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,Harry Potter and the Philosopher's Stone,fantasy young-adult fiction harry-potter ya ma...,Harry Potter and the Philosopher's Stone fanta...
2,3,"Twilight (Twilight, #1)",Twilight,young-adult fantasy vampires ya fiction parano...,Twilight young-adult fantasy vampires ya ficti...
3,4,To Kill a Mockingbird,To Kill a Mockingbird,classics classic historical-fiction school clà...,To Kill a Mockingbird classics classic histori...
4,5,The Great Gatsby,The Great Gatsby,classics fiction classic literature school his...,The Great Gatsby classics fiction classic lite...


In [15]:
# === Примеры текстовых профилей книг ===

sample_book_ids = [1, 2, 7, 10]

sample_books = books_content[
    books_content["id"].isin(sample_book_ids)
]

for _, book in sample_books.iterrows():
    print(f"\nКнига: {book['title']}")
    print(f"Подготовленное название: {book['book_title']}")
    print(f"Теги: {book['tags_text']}")
    print(f"Текстовый профиль: {book['content_text']}")


Книга: The Hunger Games (The Hunger Games, #1)
Подготовленное название: The Hunger Games
Теги: young-adult fiction dystopian dystopia fantasy ya science-fiction sci-fi romance adventure hunger-games teen post-apocalyptic the-hunger-games action suzanne-collins ya-fiction survival sci-fi-fantasy books
Текстовый профиль: The Hunger Games young-adult fiction dystopian dystopia fantasy ya science-fiction sci-fi romance adventure hunger-games teen post-apocalyptic the-hunger-games action suzanne-collins ya-fiction survival sci-fi-fantasy books

Книга: Harry Potter and the Sorcerer's Stone (Harry Potter, #1)
Подготовленное название: Harry Potter and the Philosopher's Stone
Теги: fantasy young-adult fiction harry-potter ya magic childrens children adventure classics children-s childhood middle-grade j-k-rowling sci-fi-fantasy children-s-books kids novels 5-stars i-own
Текстовый профиль: Harry Potter and the Philosopher's Stone fantasy young-adult fiction harry-potter ya magic childrens child

### Выводы по подготовке текстовых признаков

По результатам подготовки текстовых данных установлено:

1. Для всех 10 000 книг сформированы текстовые профили.
Пустых текстовых представлений не обнаружено.

2. При отсутствии `original_title` используется значение `title`,
что позволяет сохранить названия всех книг.

3. Из исходных пользовательских тегов исключены основные служебные метки,
связанные с организацией библиотеки,
повторным чтением и пользовательскими действиями.

4. Для каждой книги выбирается не более 20 наиболее часто применяемых содержательных тегов.

5. Проверка примеров показала, что текстовые профили содержат характеристики жанра,
тематики и отдельных особенностей произведений.

6. Некоторые теги остаются близкими по смыслу,
например `science-fiction` и `sci-fi`, `classics` и `classic`.
На данном этапе дополнительную нормализацию таких вариантов не выполняем.

**Итог:**

Подготовлены текстовые представления книг, объединяющие названия и пользовательские теги.

Полученные признаки можно использовать для построения TF-IDF матрицы и расчета сходства между книгами.

Качество Content-Based рекомендаций будет зависеть от информативности выбранных тегов
и особенностей их текстовой обработки.

## 2.4. TF-IDF и косинусная близость

Для построения Content-Based рекомендаций преобразуем
текстовые профили книг в числовые векторы с помощью TF-IDF.

TF-IDF учитывает частоту термина в документе и уменьшает вес слов,
которые встречаются во многих документах.

После векторизации каждая книга будет представлена вектором признаков.
Для определения сходства книг используем косинусную близость (cosine similarity).

Косинусная близость принимает значения от -1 до 1 в общем случае.
Для неотрицательных TF-IDF векторов значения находятся в диапазоне от 0 до 1:

- 1 -> одинаковое направление векторов
- 0 -> отсутствие общих ненулевых признаков;
- значения между 0 и 1 -> различная степень текстового сходства.

При этом высокий показатель сходства не обязательно означает,
что книги одинаково понравятся пользователю.

In [16]:
# === Векторизация текстовых профилей книг ===

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.95,
    norm="l2",
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    books_content["content_text"]
)

print(f"Размер TF-IDF матрицы: {tfidf_matrix.shape}")
print(f"Количество книг: {tfidf_matrix.shape[0]}")
print(f"Количество текстовых признаков: {tfidf_matrix.shape[1]}")
print(f"Количество ненулевых элементов: {tfidf_matrix.nnz}")

# Проверяем плотность полученной матрицы
tfidf_density = (
    tfidf_matrix.nnz
    / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])
    * 100
)

print(f"Плотность TF-IDF матрицы: {tfidf_density:.4f}%")

Размер TF-IDF матрицы: (10000, 5812)
Количество книг: 10000
Количество текстовых признаков: 5812
Количество ненулевых элементов: 203044
Плотность TF-IDF матрицы: 0.3494%


In [17]:
# === Проверка текстовых признаков TF-IDF ===

feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"Количество признаков в словаре: {len(feature_names)}")
print("Первые 30 признаков:")

print(feature_names[:30])

# Проверяем, что для каждой книги есть ненулевые текстовые признаки
empty_vectors_count = (tfidf_matrix.getnnz(axis=1) == 0).sum()

print(f"Количество пустых TF-IDF векторов: {empty_vectors_count}")

Количество признаков в словаре: 5812
Первые 30 признаков:
['00' '000' '007' '01' '02' '04' '10' '100' '1001' '10th' '11' '11th' '12'
 '12th' '13' '14' '14th' '15' '150' '15th' '16' '16th' '17th' '18' '1800s'
 '18th' '1920s' '1927' '1930s' '1940s']
Количество пустых TF-IDF векторов: 0


In [18]:
# === Косинусная близость книги с остальным каталогом ===

# The Hobbit
sample_book_id = 7

# Находим позицию книги в таблице текстовых профилей
book_index = books_content.index[
    books_content["id"] == sample_book_id
    ][0]

# Рассчитываем сходство только для одной книги со всем каталогом
similarity_scores = cosine_similarity(
    tfidf_matrix[book_index],
    tfidf_matrix,
).ravel()

# Исключаем исходную книгу и выбираем 10 наиболее похожих
similar_book_indices = np.argsort(similarity_scores)[::-1]
similar_book_indices = similar_book_indices[
    similar_book_indices != book_index
    ][:10]

similar_books = books_content.iloc[
    similar_book_indices
][["id", "title", "authors"]].copy()

similar_books["similarity"] = similarity_scores[similar_book_indices]

display(similar_books)


,id,title,authors,similarity
465,466,The Hobbit: Graphic Novel,"Chuck Dixon, J.R.R. Tolkien, David Wenzel, Sea...",0.774643
610,611,The Silmarillion (Middle-Earth Universe),"J.R.R. Tolkien, Christopher Tolkien, Ted Nasmith",0.748964
154,155,"The Two Towers (The Lord of the Rings, #2)",J.R.R. Tolkien,0.724484
2308,2309,The Children of Húrin,"J.R.R. Tolkien, Christopher Tolkien, Alan Lee",0.723397
160,161,"The Return of the King (The Lord of the Rings,...",J.R.R. Tolkien,0.720150
188,189,"The Lord of the Rings (The Lord of the Rings, ...",J.R.R. Tolkien,0.668904
18,19,The Fellowship of the Ring (The Lord of the Ri...,J.R.R. Tolkien,0.647958
963,964,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,J.R.R. Tolkien,0.627834
4975,4976,Unfinished Tales of Númenor and Middle-Earth,"J.R.R. Tolkien, Christopher Tolkien",0.626078
8271,8272,The Complete Guide to Middle-Earth,"Robert Foster, J.R.R. Tolkien",0.549629


### Выводы по TF-IDF и косинусной близости

По результатам построения TF-IDF матрицы и расчета сходства книг установлено:

1. Для 10 000 книг сформирована TF-IDF матрица размерностью 10 000 x 5 812.
Количество текстовых признаков составляет 5 812.

2. Плотность матрицы составляет 0,3494%, что подтверждает разреженность текстового представления.
Для хранения и вычислений используется разреженная матрица.

3. Пустых TF-IDF векторов не обнаружено.
Все книги сохранили текстовые признаки после векторизации.

4. Косинусная близость позволяет находить книги с похожими текстовыми характеристиками
на основе названий и пользовательских тегов.

5. Для книги `The Hobbit` наиболее похожими оказались произведения Дж.Р.Р. Толкина
и другие издания, связанные со вселенной Средиземья.
Максимальное значение сходства составило 0,7746.

6. Полученные рекомендации демонстрируют тематическую близость,
однако имеют ограниченное разнообразие, так как большинство произведений относятся к одному автору или литературной вселенной.

**Итог:**

TF-IDF и косинусная близость позволяют построить Content-Based рекомендации без использования пользовательских оценок.

Метод хорошо выявляет сходство по текстовым характеристикам,
но не учитывает индивидуальные предпочтения пользователя
и может рекомендовать слишком похожие произведения.

## 2.5. Получение похожих книг

Реализована функция `get_similar_books()`,
которая возвращает Top-N наиболее похожих книг по внутреннему идентификатору.

Алгоритм выполняет следующие действия:

1. Находит исходную книгу в матрице TF-IDF.
2. Рассчитывает косинусную близость с остальными книгами.
3. Исключает исходную книгу из результатов.
4. Сортирует книги по убыванию сходства.
5. Возвращает заданное количество рекомендаций.

Проверим количество рекомендаций, отсутствие исходной книги и корректность сортировки.

In [19]:
# === Проверка функции получения похожих книг ===

book_id = 7  # The Hobbit

recommendations = get_similar_books(
    book_id=book_id,
    books_content=books_content,
    tfidf_matrix=tfidf_matrix,
    top_n=10,
)

print(f"Количество рекомендаций: {len(recommendations)}")
print(
    "Исходная книга присутствует в рекомендациях:",
    book_id in recommendations["id"].values,
)

# Проверяем отсутствие исходной книги и корректность порядка сходства
assert book_id not in recommendations["id"].values
assert recommendations["similarity"].is_monotonic_decreasing

display(recommendations)

Количество рекомендаций: 10
Исходная книга присутствует в рекомендациях: False


,id,title,authors,similarity
0,466,The Hobbit: Graphic Novel,"Chuck Dixon, J.R.R. Tolkien, David Wenzel, Sea...",0.774643
1,611,The Silmarillion (Middle-Earth Universe),"J.R.R. Tolkien, Christopher Tolkien, Ted Nasmith",0.748964
2,155,"The Two Towers (The Lord of the Rings, #2)",J.R.R. Tolkien,0.724484
3,2309,The Children of Húrin,"J.R.R. Tolkien, Christopher Tolkien, Alan Lee",0.723397
4,161,"The Return of the King (The Lord of the Rings,...",J.R.R. Tolkien,0.720150
5,189,"The Lord of the Rings (The Lord of the Rings, ...",J.R.R. Tolkien,0.668904
6,19,The Fellowship of the Ring (The Lord of the Ri...,J.R.R. Tolkien,0.647958
7,964,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,J.R.R. Tolkien,0.627834
8,4976,Unfinished Tales of Númenor and Middle-Earth,"J.R.R. Tolkien, Christopher Tolkien",0.626078
9,8272,The Complete Guide to Middle-Earth,"Robert Foster, J.R.R. Tolkien",0.549629


## 2.6. Примеры рекомендаций и их интерпретация

Проверим Content-Based модель на книгах разных жанров:

- `The Hunger Games` -> антиутопия и приключенческая фантастика
- `Harry Potter and the Sorcerer's Stone` -> фэнтези
- `Pride and Prejudice` -> классическая литература и романтическая проза

Сравним рекомендации и оценим, насколько они соответствуют тематике исходных произведений.

In [20]:
# === Примеры Content-Based рекомендаций для разных книг ===

sample_book_ids = [1, 2, 10]

for book_id in sample_book_ids:
    book_title = books_content.loc[
        books_content["id"] == book_id,
        "title",
    ].iloc[0]

    recommendations = get_similar_books(
        book_id=book_id,
        books_content=books_content,
        tfidf_matrix=tfidf_matrix,
        top_n=5,
    )

    print(f"\nИсходная книга: {book_title}")

    display(recommendations)


Исходная книга: The Hunger Games (The Hunger Games, #1)


,id,title,authors,similarity
0,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.956951
1,507,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,0.928419
2,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.892182
3,6224,The World of the Hunger Games (Hunger Games Tr...,Kate Egan,0.889254
4,717,The Hunger Games: Official Illustrated Movie C...,Kate Egan,0.791987



Исходная книга: Harry Potter and the Sorcerer's Stone (Harry Potter, #1)


,id,title,authors,similarity
0,24,Harry Potter and the Goblet of Fire (Harry Pot...,"J.K. Rowling, Mary GrandPré",0.928528
1,18,Harry Potter and the Prisoner of Azkaban (Harr...,"J.K. Rowling, Mary GrandPré, Rufus Beck",0.912851
2,25,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré",0.901664
3,23,Harry Potter and the Chamber of Secrets (Harry...,"J.K. Rowling, Mary GrandPré",0.862126
4,2101,"The Harry Potter Collection 1-4 (Harry Potter,...","J.K. Rowling, Mary GrandPré",0.851308



Исходная книга: Pride and Prejudice


,id,title,authors,similarity
0,171,Emma,"Jane Austen, Fiona Stafford",0.758717
1,230,Persuasion,"Jane Austen, James Kinsley, Deidre Shauna Lynch",0.741008
2,63,Wuthering Heights,"Emily Brontë, Richard J. Dunn",0.700719
3,471,Mansfield Park,"Jane Austen, Kathryn Sutherland, Tony Tanner, ...",0.696735
4,451,Northanger Abbey,"Jane Austen, Alfred MacAdam",0.677227


### Выводы по Content-Based рекомендациям

По результатам проверки алгоритма установлено:

1. Функция `get_similar_books()` корректно возвращает заданное количество похожих книг,
исключает исходное произведение и сортирует рекомендации по убыванию косинусной близости.

2. Для `The Hobbit` наиболее похожими оказались произведения Дж.Р.Р. Толкина и книги,
связанные со вселенной Средиземья.

3. Для `The Hunger Games` алгоритм рекомендует продолжения серии и дополнительные издания.
Максимальное значение сходства составляет 0,9570.

4. Для `Harry Potter and the Sorcerer's Stone` наиболее похожими оказались другие части серии о Гарри Поттере.
Максимальное сходство составляет 0,9285.

5. Для `Pride and Prejudice` рекомендации включают произведения Джейн Остин
и другие тематически близкие книги классической литературы.

6. Алгоритм хорошо определяет текстовое сходство, однако часто рекомендует произведения одного автора или одной серии.
Это ограничивает разнообразие рекомендаций.

7. Косинусная близость характеризует сходство текстовых признаков,
но не является оценкой вероятности того, что книга понравится пользователю.

**Итог:**

Реализована Content-Based модель, которая формирует рекомендации на основе названий книг и пользовательских тегов.

Полученные результаты демонстрируют содержательную близость рекомендуемых произведений.
При этом алгоритм не использует историю оценок пользователя и не обеспечивает персонализацию самостоятельно.

Для построения персональных рекомендаций в дальнейшем потребуется учитывать историю взаимодействий пользователя с книгами.

## 2.7. Итоговые выводы по этапу 2

В ходе второго этапа реализованы два базовых подхода к построению книжных рекомендаций:
- Popularity Baseline
- Content-Based Filtering

**1. Popularity Baseline**

- Рейтинг книг формируется на основе средней пользовательской оценки с минимальным порогом количества оценок.
- При пороге в 20 оценок в рейтинге осталось 9 997 книг.
- Для конкретного пользователя реализовано исключение ранее оцененных книг.
- Алгоритм прост в реализации, но не учитывает индивидуальные предпочтения пользователей.

**2. Content-Based Filtering**

- Для всех 10 000 книг подготовлены текстовые профили из названий и пользовательских тегов.
- Для векторизации использован TF-IDF, сформирована матрица размерностью 10 000 x 5 812.
- Сходство книг рассчитывается с помощью косинусной близости.
- Реализована функция `get_similar_books()` для получения Top-N похожих книг.
- Проверка на нескольких произведениях показала, что алгоритм успешно находит тематически близкие книги.

**3. Сравнение подходов**

Popularity Baseline формирует общий рейтинг книг и может использоваться при отсутствии пользовательской истории.

Content-Based Filtering учитывает характеристики произведений и позволяет находить похожие книги,
но без дополнительной обработки пользовательской истории не формирует персонализированную выдачу.

Оба подхода имеют ограничения и могут использоваться как базовые алгоритмы для сравнения с более сложными рекомендательными моделями.

**Общий вывод:**

На втором этапе построены и проверены базовая рейтинговая и контентная рекомендательные модели.